In [1]:
# ============================================================
# CELL 1 (03_feature_engineering, replaces the old Cells 1-2): Rebuild the base table
# What it does:
#   1) setup that survives a kernel restart: imports, paths, rulebook
#      (eda_decisions.json), station_meta, aligned rainfall, flags, satellite
#      features (all from disk; no Earth Engine needed)
#   2) applies the trace rule (< 0.1 mm -> 0) to the aligned rainfall and
#      saves data/processed/rain_final.parquet; clears the Sikles flag on
#      2009-08-29 (that value moved to 08-30 in the date correction)
#   3) turns the 9 ERA5-Land level variables into anomalies from each
#      location's own day-of-year climatology (training years <= 2010,
#      31-day circular smoothing); rainfall features and pressure change stay
#      as they are; saves data/satellite/satellite_features_model.parquet
#   4) builds the long base table for the 7 gauged stations (rain target,
#      flag, calendar, static site info, period) and merges the satellite
#      features; saves data/processed/model_base.parquet
#   5) prints: row counts, valid-target days and wet-day share by station
#      and period, satellite feature count and missing values on valid rows,
#      and the station-identity check (spread of station means, raw vs anomaly)
# Chapakot's satellite features stay in satellite_features_model.parquet
# ============================================================
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 150)

ROOT = Path.cwd().parent
PROCESSED_DIR = ROOT / "data" / "processed"
SAT_DIR = ROOT / "data" / "satellite"

# --- 1) load ---
with open(PROCESSED_DIR / "eda_decisions.json") as f:
    dec = json.load(f)
WET = dec["wet_day_mm"]
SEASON_OF_MONTH = {int(k): v for k, v in dec["seasons"].items()}
TRAIN_END = pd.Timestamp(dec["temporal_split"]["train"][1])

meta = pd.read_csv(PROCESSED_DIR / "station_meta.csv", index_col="station")
stations = list(meta.index)
rain_a = pd.read_parquet(PROCESSED_DIR / "rain_wide_aligned.parquet")[stations]
flags = pd.read_parquet(PROCESSED_DIR / "stations_flags.parquet")[stations].copy()
sat = pd.read_parquet(SAT_DIR / "satellite_features.parquet")

# --- 2) trace rule, flags ---
trace = (rain_a > 0) & (rain_a < 0.1)
rain = rain_a.mask(trace, 0.0)
print("Trace values set to 0:", int(trace.sum().sum()))
rain.to_parquet(PROCESSED_DIR / "rain_final.parquet")

flags.loc["2009-08-29", "Sikles"] = False
flags = flags & rain.notna()

# --- 3) anomalies for the level variables ---
LEVEL_VARS = ["era_t2m_c", "era_dtr_c", "era_dewdep_c", "era_rh_pct", "era_u10",
              "era_v10", "era_wspd", "era_sp_hpa", "era_soil_w1"]
frames = []
for st, d in sat.groupby("station"):
    d = d.sort_values("date").set_index("date")
    doy = d.index.dayofyear
    tr = d.index <= TRAIN_END
    clim = d.loc[tr, LEVEL_VARS].groupby(d.index[tr].dayofyear).mean()
    clim = clim.reindex(range(1, 367)).interpolate(limit_direction="both")
    ext = pd.concat([clim.iloc[-15:], clim, clim.iloc[:15]])
    sm = ext.rolling(31, center=True).mean().iloc[15:-15]
    sm.index = range(1, 367)
    a = d[LEVEL_VARS] - sm.reindex(doy).to_numpy()
    a.columns = ["anom_" + c for c in LEVEL_VARS]
    a.insert(0, "station", st)
    frames.append(a.reset_index())
anom = pd.concat(frames, ignore_index=True)

keep = [c for c in sat.columns if c not in LEVEL_VARS and c not in ("station", "date")]
sat_feat = sat[["station", "date"] + keep].merge(anom, on=["station", "date"])
sat_feat.to_parquet(SAT_DIR / "satellite_features_model.parquet")
feat_cols = [c for c in sat_feat.columns if c not in ("station", "date")]

# --- 4) long base table for the 7 gauged stations ---
doy_g = rain.index.dayofyear.values
parts = []
for st in stations:
    parts.append(pd.DataFrame({
        "date": rain.index, "station": st,
        "rain": rain[st].values, "flagged": flags[st].values,
        "doy_sin": np.sin(2 * np.pi * doy_g / 365.25),
        "doy_cos": np.cos(2 * np.pi * doy_g / 365.25),
        "month": rain.index.month.values,
        "season": rain.index.month.map(SEASON_OF_MONTH).values,
        "lat": meta.loc[st, "lat"], "lon": meta.loc[st, "lon"],
        "elev_m": meta.loc[st, "elev_m"],
    }))
base = pd.concat(parts, ignore_index=True)
base["period"] = np.where(base["date"] <= TRAIN_END, "train", "test")
base = base.merge(sat_feat, on=["station", "date"], how="left")
base.to_parquet(PROCESSED_DIR / "model_base.parquet")

# --- 5) report ---
print("\nBase table:", base.shape, "| satellite feature columns:", len(feat_cols))
valid = base.dropna(subset=["rain"])
cnt = valid.pivot_table(index="station", columns="period", values="rain", aggfunc="count").loc[stations]
wetp = (valid.assign(wet=valid["rain"] >= WET)
        .pivot_table(index="station", columns="period", values="wet", aggfunc="mean")
        .mul(100).round(1).loc[stations])
print("\nValid target days by station and period:")
print(cnt.to_string())
print("\nWet-day share (%) by station and period:")
print(wetp.to_string())
print("\nRows with a valid target:", len(valid),
      "| of which with any satellite feature missing:",
      int(valid[feat_cols].isna().any(axis=1).sum()))

print("\nStation-identity check: spread (std) across the 8 locations of the station means")
rows = []
for v in ["era_t2m_c", "era_rh_pct", "era_sp_hpa", "era_soil_w1"]:
    rows.append({"variable": v,
                 "raw_spread": round(sat.groupby("station")[v].mean().std(), 3),
                 "anomaly_spread": round(sat_feat.groupby("station")["anom_" + v].mean().std(), 3)})
print(pd.DataFrame(rows).to_string(index=False))

per = np.where(sat_feat["date"] <= TRAIN_END, "train", "test")
print("\nMean anomaly by period (all 8 locations):")
print(sat_feat.groupby(per)[["anom_era_t2m_c", "anom_era_rh_pct", "anom_era_sp_hpa", "anom_era_soil_w1"]]
      .mean().round(2).to_string())

Trace values set to 0: 0

Base table: (104825, 36) | satellite feature columns: 24

Valid target days by station and period:
period           test  train
station                     
Pokhara Airport  4748  10227
Kunchha          4564  10131
Bandipur         4260   9280
Lumle            4717  10192
Damauli          4412   9831
Sikles           4715  10224
Lamachaur        4717   9616

Wet-day share (%) by station and period:
period           test  train
station                     
Pokhara Airport  38.9   39.6
Kunchha          27.7   26.7
Bandipur         27.7   25.7
Lumle            46.0   46.9
Damauli          22.9   28.0
Sikles           50.0   50.0
Lamachaur        40.8   40.2

Rows with a valid target: 101634 | of which with any satellite feature missing: 25

Station-identity check: spread (std) across the 8 locations of the station means
   variable  raw_spread  anomaly_spread
  era_t2m_c       5.351           0.036
 era_rh_pct       3.799           0.119
 era_sp_hpa      83.088  

In [2]:
# ============================================================
# CELL 2 (03_feature_engineering): Shared code in src/ (metrics + splits) and a smoke test
# What it does:
#   1) writes src/metrics.py and src/splits.py (see the docstrings in the files)
#   2) imports them and runs checks on synthetic data with known answers:
#      perfect simulation -> KGE 1, NSE 1, bias 0, POD 1, FAR 0;
#      simulation = 2 x observation -> bias +100 %, KGE = 1 - sqrt(2) = -0.414
#   3) rebuilds the blocked folds from station_meta and asserts that they
#      equal the ones in eda_decisions.json; prints, per fold, how many
#      stations train and how many rows fall in each of the four masks
#   4) smoke test on real data: scores raw CHIRPS (chirps_blend_c) against
#      the gauges on the test years (2011-2023), per station
# ============================================================
import importlib
import json
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 170)

ROOT = Path.cwd().parent
PROCESSED_DIR = ROOT / "data" / "processed"
SRC = ROOT / "src"
SRC.mkdir(exist_ok=True)

METRICS_CODE = r'''"""Shared scoring functions for the rainfall project."""
import numpy as np
import pandas as pd

WET = 1.0


def _pair(obs, sim):
    obs = np.asarray(obs, dtype=float)
    sim = np.asarray(sim, dtype=float)
    ok = np.isfinite(obs) & np.isfinite(sim)
    return obs[ok], sim[ok]


def pearson_r(obs, sim):
    o, s = _pair(obs, sim)
    if len(o) < 3 or o.std() == 0 or s.std() == 0:
        return np.nan
    return float(np.corrcoef(o, s)[0, 1])


def rmse(obs, sim):
    o, s = _pair(obs, sim)
    return float(np.sqrt(np.mean((s - o) ** 2))) if len(o) else np.nan


def mae(obs, sim):
    o, s = _pair(obs, sim)
    return float(np.mean(np.abs(s - o))) if len(o) else np.nan


def pbias(obs, sim):
    """Percent bias of the total: 100 * (sum(sim) - sum(obs)) / sum(obs)."""
    o, s = _pair(obs, sim)
    return float(100 * (s.sum() - o.sum()) / o.sum()) if len(o) and o.sum() > 0 else np.nan


def nse(obs, sim):
    o, s = _pair(obs, sim)
    den = ((o - o.mean()) ** 2).sum() if len(o) else 0
    return float(1 - ((s - o) ** 2).sum() / den) if den > 0 else np.nan


def kge(obs, sim):
    """Kling-Gupta efficiency (Gupta et al. 2009): 1 - sqrt((r-1)^2 + (alpha-1)^2 + (beta-1)^2)."""
    o, s = _pair(obs, sim)
    if len(o) < 3 or o.std() == 0 or o.mean() == 0 or s.std() == 0:
        return np.nan
    r = np.corrcoef(o, s)[0, 1]
    alpha = s.std() / o.std()
    beta = s.mean() / o.mean()
    return float(1 - np.sqrt((r - 1) ** 2 + (alpha - 1) ** 2 + (beta - 1) ** 2))


def categorical(obs, sim, thr=WET):
    """Detection of days at or above thr: POD, FAR, CSI and frequency bias."""
    o, s = _pair(obs, sim)
    ob, sb = o >= thr, s >= thr
    hits = int((ob & sb).sum())
    miss = int((ob & ~sb).sum())
    fa = int((~ob & sb).sum())
    return {
        "pod": hits / (hits + miss) if hits + miss else np.nan,
        "far": fa / (hits + fa) if hits + fa else np.nan,
        "csi": hits / (hits + miss + fa) if hits + miss + fa else np.nan,
        "freq_bias": (hits + fa) / (hits + miss) if hits + miss else np.nan,
    }


def wet_quantile_ratio(obs, sim, q, thr=WET, min_days=50):
    """Simulated wet-day quantile divided by the observed one."""
    o, s = _pair(obs, sim)
    ow, sw = o[o >= thr], s[s >= thr]
    if len(ow) < min_days or len(sw) < min_days:
        return np.nan
    return float(np.quantile(sw, q) / np.quantile(ow, q))


def rx1day(dates, obs, sim, min_days=300):
    """Annual maximum 1-day rainfall: ratio of means (sim/obs) and correlation across years."""
    df = pd.DataFrame({"date": pd.to_datetime(dates),
                       "obs": np.asarray(obs, dtype=float),
                       "sim": np.asarray(sim, dtype=float)}).dropna()
    empty = {"rx1_years": 0, "rx1_ratio": np.nan, "rx1_r": np.nan}
    if df.empty:
        return empty
    g = df.groupby(df["date"].dt.year)
    n = g.size()
    mx = g[["obs", "sim"]].max()[n >= min_days]
    if len(mx) < 3:
        return empty
    return {"rx1_years": int(len(mx)),
            "rx1_ratio": float(mx["sim"].mean() / mx["obs"].mean()),
            "rx1_r": float(mx["obs"].corr(mx["sim"])) if len(mx) >= 5 else np.nan}


def score_all(dates, obs, sim, thr=None, fixed_extreme=50.0, wet=WET):
    """All scores for one station. thr = {'P95_wet': x, 'P99_wet': y} (station-specific)."""
    o, s = _pair(obs, sim)
    if len(o) == 0:
        return {"n": 0}
    out = {"n": int(len(o)), "mean_obs": float(o.mean()), "mean_sim": float(s.mean()),
           "r": pearson_r(o, s), "kge": kge(o, s), "nse": nse(o, s),
           "rmse": rmse(o, s), "mae": mae(o, s), "pbias": pbias(o, s),
           "wet_freq_obs": float(100 * (o >= wet).mean()),
           "wet_freq_sim": float(100 * (s >= wet).mean())}
    for k, v in categorical(o, s, wet).items():
        out["wet_" + k] = v
    out["q95_ratio"] = wet_quantile_ratio(o, s, 0.95, wet)
    out["q99_ratio"] = wet_quantile_ratio(o, s, 0.99, wet)
    out.update(rx1day(dates, obs, sim))
    levels = [("p95", thr["P95_wet"]), ("p99", thr["P99_wet"])] if thr else []
    levels.append(("fix", fixed_extreme))
    for name, val in levels:
        for k, v in categorical(o, s, val).items():
            out[name + "_" + k] = v
    return out


def score_table(df, thresholds=None, fixed_extreme=50.0):
    """df needs columns date, station, obs, sim. Returns one row of scores per station."""
    rows = {}
    for st, d in df.groupby("station"):
        thr = (thresholds or {}).get(st)
        rows[st] = score_all(d["date"].values, d["obs"].values, d["sim"].values,
                             thr, fixed_extreme)
    return pd.DataFrame(rows).T
'''

SPLITS_CODE = r'''"""Spatial and temporal splits for the rainfall project."""
import numpy as np
import pandas as pd


def distance_matrix(meta):
    """Great-circle distances (km) between stations; meta needs lat and lon columns."""
    R = 6371.0
    lat = np.radians(meta["lat"].values)
    lon = np.radians(meta["lon"].values)
    a = (np.sin((lat[:, None] - lat[None, :]) / 2) ** 2
         + np.cos(lat)[:, None] * np.cos(lat)[None, :]
         * np.sin((lon[:, None] - lon[None, :]) / 2) ** 2)
    return pd.DataFrame(2 * R * np.arcsin(np.sqrt(a)), index=meta.index, columns=meta.index)


def make_folds(stations, dist, scheme="blocked", block_km=30):
    """One fold per held-out station. 'loso' trains on all other stations;
    'blocked' also removes every station closer than block_km to the held-out one."""
    folds = []
    for h in stations:
        if scheme == "loso":
            train = [s for s in stations if s != h]
        elif scheme == "blocked":
            train = [s for s in stations if s != h and dist.loc[h, s] >= block_km]
        else:
            raise ValueError("scheme must be 'loso' or 'blocked'")
        folds.append({"held_out": h, "train": train, "scheme": scheme})
    return folds


def fold_masks(df, fold, train_end):
    """Boolean row masks for a long table with columns station and date.
    train          : training stations, years <= train_end
    temporal_only  : training stations, years  > train_end
    spatial_only   : held-out station, years <= train_end
    headline       : held-out station, years  > train_end  (unseen site AND unseen years)"""
    early = (df["date"] <= pd.Timestamp(train_end)).values
    in_train = df["station"].isin(fold["train"]).values
    held = (df["station"] == fold["held_out"]).values
    return {"train": in_train & early, "temporal_only": in_train & ~early,
            "spatial_only": held & early, "headline": held & ~early}
'''

(SRC / "metrics.py").write_text(METRICS_CODE, encoding="utf-8")
(SRC / "splits.py").write_text(SPLITS_CODE, encoding="utf-8")
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
import metrics, splits
importlib.reload(metrics)
importlib.reload(splits)
print("Wrote and imported src/metrics.py and src/splits.py")

# --- 2) synthetic checks ---
rng = np.random.default_rng(0)
obs = rng.exponential(5, 5000) * (rng.random(5000) < 0.5)
d = pd.date_range("2000-01-01", periods=5000)
perfect = metrics.score_all(d, obs, obs.copy())
double = metrics.score_all(d, obs, 2 * obs)
assert abs(perfect["kge"] - 1) < 1e-9 and abs(perfect["nse"] - 1) < 1e-9
assert abs(perfect["pbias"]) < 1e-9 and perfect["wet_pod"] == 1 and perfect["wet_far"] == 0
assert abs(double["pbias"] - 100) < 1e-9 and abs(double["kge"] - (1 - np.sqrt(2))) < 1e-9
print("Synthetic metric checks passed (perfect: KGE 1; doubled: bias +100 %, KGE -0.414)")

# --- 3) folds ---
with open(PROCESSED_DIR / "eda_decisions.json") as f:
    dec = json.load(f)
meta = pd.read_csv(PROCESSED_DIR / "station_meta.csv", index_col="station")
stations = list(meta.index)
dist = splits.distance_matrix(meta)
TRAIN_END = dec["temporal_split"]["train"][1]

blocked = splits.make_folds(stations, dist, "blocked", dec["spatial_split"]["block_km"])
loso = splits.make_folds(stations, dist, "loso")
for fo in blocked:
    assert sorted(fo["train"]) == sorted(dec["spatial_split"]["blocked_train_stations"][fo["held_out"]])
print("Blocked folds match eda_decisions.json")

base = pd.read_parquet(PROCESSED_DIR / "model_base.parquet")
valid = base.dropna(subset=["rain"]).reset_index(drop=True)
rows = []
for fo in blocked:
    m = splits.fold_masks(valid, fo, TRAIN_END)
    rows.append({"held_out": fo["held_out"], "n_train_stn": len(fo["train"]),
                 **{k: int(v.sum()) for k, v in m.items()}})
print("\nBlocked folds: rows with a valid target in each mask")
print(pd.DataFrame(rows).to_string(index=False))

# --- 4) smoke test: raw CHIRPS on the test years ---
t = valid.dropna(subset=["chirps_blend_c"])
t = t[t["period"] == "test"]
df = pd.DataFrame({"date": t["date"], "station": t["station"],
                   "obs": t["rain"], "sim": t["chirps_blend_c"]})
sc = metrics.score_table(df, thresholds=dec["extreme_thresholds_mm"])
cols = ["n", "r", "kge", "pbias", "wet_pod", "wet_far", "wet_csi", "wet_freq_obs",
        "wet_freq_sim", "q95_ratio", "q99_ratio", "rx1_ratio", "p95_pod", "p95_far"]
print("\nSmoke test: raw CHIRPS blend vs gauge, test years 2011-2023, per station")
print(sc[cols].astype(float).round(2).to_string())

Wrote and imported src/metrics.py and src/splits.py
Synthetic metric checks passed (perfect: KGE 1; doubled: bias +100 %, KGE -0.414)
Blocked folds match eda_decisions.json

Blocked folds: rows with a valid target in each mask
       held_out  n_train_stn  train  temporal_only  spatial_only  headline
Pokhara Airport            3  29242          13236         10227      4748
        Kunchha            4  40259          18897         10131      4564
       Bandipur            4  40259          18897          9280      4260
          Lumle            3  29242          13236         10192      4717
        Damauli            4  40259          18897          9831      4412
         Sikles            3  29242          13236         10224      4715
      Lamachaur            3  29242          13236          9616      4717

Smoke test: raw CHIRPS blend vs gauge, test years 2011-2023, per station
                      n     r   kge  pbias  wet_pod  wet_far  wet_csi  wet_freq_obs  wet_freq_sim  